In [1]:
import os
import json
import cv2
import numpy as np
from scipy.signal import find_peaks
from tqdm.auto import tqdm

# --- 1. IMPORTY ---
from software._load_image import load_image
from software.preprocessing.image_aligner.detection import extract_long_lines, filter_horizontal_lines
from software.preprocessing.image_aligner.geometry import calculate_alpha, rad_to_deg
from software.preprocessing.image_aligner.transform import rotate_image_full_alpha
from software.masks.generate import create_mask_from_json

# --- 2. LOKALNA KONFIGURACJA ŚCIEŻEK ---
# Pobiera dane z Twojego lokalnego folderu 'data'
SOURCE_TRAIN_DIR = 'data/train/'
SOURCE_VAL_DIR = 'data/val/'

# Zapisuje gotowe łatki do nowych folderów
TARGET_TRAIN_DIR = 'data/train_patches/'
TARGET_VAL_DIR = 'data/val_patches/'

TARGET_RES = 512
BEST_OY = 25
BEST_OX = 0

LEAD_GRID = [
    ["I", "aVR", "V1", "V4"],
    ["II", "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"]
]

# Tworzenie struktury folderów docelowych
for subset in [TARGET_TRAIN_DIR, TARGET_VAL_DIR]:
    os.makedirs(os.path.join(subset, 'images'), exist_ok=True)
    os.makedirs(os.path.join(subset, 'masks'), exist_ok=True)

# --- 3. GŁÓWNA FUNKCJA GENERUJĄCA ---
def process_folder(source_dir, target_dir):
    print(f"\nPrzetwarzam folder: {source_dir}")
    json_files = sorted([f for f in os.listdir(source_dir) if f.endswith('.json')])

    for json_name in tqdm(json_files):
        base_name = json_name.replace('.json', '')

        # Pomiń, jeśli plik już istnieje (przydatne przy przerywaniu i wznawianiu)
        check_file = os.path.join(target_dir, 'images', f"{base_name}_lead_I.png")
        if os.path.exists(check_file):
            continue

        img_path = os.path.join(source_dir, base_name + '.png')
        if not os.path.exists(img_path): img_path = os.path.join(source_dir, base_name + '.jpg')

        img_raw = load_image(img_path)
        img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB) if img_raw.shape[2] == 3 else img_raw
        orig_h, orig_w = img_raw.shape[:2]

        _, lines = extract_long_lines(img_raw)
        filtered = filter_horizontal_lines(lines)
        angle = rad_to_deg(calculate_alpha(filtered)) if filtered else 0.0

        mask_raw = create_mask_from_json(os.path.join(source_dir, json_name), (orig_h, orig_w), apply_angle=angle)
        if len(mask_raw.shape) > 2: mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2GRAY)
        _, mask_bin = cv2.threshold(mask_raw, 127, 255, cv2.THRESH_BINARY)
        if np.mean(mask_bin) > 127: mask_bin = cv2.bitwise_not(mask_bin)
        mask_raw_3ch = cv2.cvtColor(mask_bin, cv2.COLOR_GRAY2RGB)

        img_straight = rotate_image_full_alpha(img_raw, angle)
        if img_straight.shape[2] == 4: img_straight = cv2.cvtColor(img_straight, cv2.COLOR_RGBA2RGB)

        mask_straight_3ch = rotate_image_full_alpha(mask_raw_3ch, angle)
        if mask_straight_3ch.shape[2] == 4: mask_straight_3ch = cv2.cvtColor(mask_straight_3ch, cv2.COLOR_RGBA2RGB)

        mask_straight = mask_straight_3ch[:, :, 0]
        mask_straight = (mask_straight > 127).astype(np.uint8) * 255
        new_h, new_w = mask_straight.shape

        y_profile = mask_straight.sum(axis=1)
        x_sums = mask_straight.sum(axis=0)

        peaks, _ = find_peaks(y_profile, distance=new_h//10, prominence=5)
        num_rows = len(peaks) if len(peaks) >= 3 else 4

        active_y = np.where(y_profile > 5)[0]
        active_x = np.where(x_sums > 5)[0]

        y_start_base = active_y[0] if len(active_y) > 0 else 0
        x_start_base = active_x[0] if len(active_x) > 0 else 0

        row_h_base = ((active_y[-1] - active_y[0]) if len(active_y) > 0 else new_h) / num_rows
        col_w_base = ((active_x[-1] - active_x[0]) if len(active_x) > 0 else new_w) / 4

        y_start = y_start_base + BEST_OY
        x_start = x_start_base + BEST_OX

        for r_idx in range(3):
            for c_idx in range(4):
                y1 = max(0, int(y_start + r_idx * row_h_base))
                y2 = min(new_h, int(y_start + (r_idx + 1) * row_h_base))
                x1 = max(0, int(x_start + c_idx * col_w_base))
                x2 = min(new_w, int(x_start + (c_idx + 1) * col_w_base))

                if y1 >= y2 or x1 >= x2: continue

                img_patch = img_straight[y1:y2, x1:x2]
                mask_patch = mask_straight[y1:y2, x1:x2]

                img_final = cv2.resize(img_patch, (TARGET_RES, TARGET_RES), interpolation=cv2.INTER_AREA)
                mask_final = cv2.resize(mask_patch, (TARGET_RES, TARGET_RES), interpolation=cv2.INTER_NEAREST)

                lead_name = LEAD_GRID[r_idx][c_idx]
                img_save_path = os.path.join(target_dir, 'images', f"{base_name}_lead_{lead_name}.png")
                mask_save_path = os.path.join(target_dir, 'masks', f"{base_name}_lead_{lead_name}.png")

                cv2.imwrite(img_save_path, cv2.cvtColor(img_final, cv2.COLOR_RGB2BGR))
                cv2.imwrite(mask_save_path, mask_final)

# --- 4. URUCHOMIENIE ---
process_folder(SOURCE_TRAIN_DIR, TARGET_TRAIN_DIR)
process_folder(SOURCE_VAL_DIR, TARGET_VAL_DIR)
print("\nZakończono generowanie wszystkich wycinków!")


Przetwarzam folder: data/train/


  0%|          | 0/2800 [00:00<?, ?it/s]


Przetwarzam folder: data/val/


  0%|          | 0/200 [00:00<?, ?it/s]


Zakończono generowanie wszystkich wycinków!
